# Treinar a voz da Sexta-feira (Piper)

Este caderno transforma as gravações do **Estúdio de voz** numa voz Piper (`.onnx`) que roda offline no app, sem placa de vídeo.

Antes de começar:
1. No menu **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha **GPU (T4)**.
2. Compacte a pasta `Documentos\Lumi\estudio-voz` (com `wavs` e `metadata.csv` dentro) num arquivo **`estudio-voz.zip`** e envie para o seu Google Drive, na pasta **`Lumi`** (ou seja: `Meu Drive/Lumi/estudio-voz.zip`).
3. Rode as células **na ordem**, uma de cada vez (botão ▶ de cada célula).

O treino parte da voz brasileira oficial do Piper (*faber*, medium) e aprende o timbre das suas gravações. O progresso fica salvo no Drive: se o Colab desconectar, rode tudo de novo que ele **continua de onde parou**.

In [ ]:
#@title 1. Configurações
NOME_VOZ = "sexta-feira"   #@param {type:"string"}
EPOCAS = 1000              #@param {type:"integer"}
LOTE = 16                  #@param {type:"integer"}
PASTA_DRIVE = "/content/drive/MyDrive/Lumi"
ZIP = f"{PASTA_DRIVE}/estudio-voz.zip"
TREINO = f"{PASTA_DRIVE}/treino-{NOME_VOZ}"
print("Voz:", NOME_VOZ, "| épocas:", EPOCAS, "| lote:", LOTE)

In [ ]:
#@title 2. Conferir a placa de vídeo e conectar o Google Drive
!nvidia-smi -L || echo "⚠️ Sem GPU! Mude o tipo de ambiente de execução para GPU."
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.exists(ZIP), f"Não achei {ZIP}. Envie o estudio-voz.zip para Meu Drive/Lumi/."
os.makedirs(TREINO, exist_ok=True)
print("OK, achei", ZIP)

In [ ]:
#@title 3. Instalar o Piper (leva uns minutos)
!apt-get -qq update && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
%cd /content
!test -d piper1-gpl || git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace -q
print("Piper instalado.")

In [ ]:
#@title 4. Preparar as gravações
import zipfile, glob, wave, os
DADOS = "/content/dados"
!rm -rf {DADOS} && mkdir -p {DADOS}
zipfile.ZipFile(ZIP).extractall(DADOS)
meta = glob.glob(f"{DADOS}/**/metadata.csv", recursive=True)
assert meta, "O zip não tem metadata.csv"
PASTA = os.path.dirname(meta[0]); WAVS = os.path.join(PASTA, "wavs")
linhas = [l for l in open(meta[0], encoding="utf-8").read().splitlines() if "|" in l]
seg = 0.0
for l in linhas:
    w = wave.open(os.path.join(WAVS, l.split("|")[0])); seg += w.getnframes() / w.getframerate(); w.close()
print(f"{len(linhas)} frases, {seg/60:.1f} minutos de fala.")
if seg < 5*60: print("⚠️ Pouco áudio: grave mais frases no Estúdio para ficar melhor.")

In [ ]:
#@title 5. Baixar a voz brasileira de partida (faber, 849 MB)
from huggingface_hub import hf_hub_download
BASE = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset",
                       filename="pt/pt_BR/faber/medium/epoch=1999-step=92000.ckpt")
print("Base:", BASE)

In [ ]:
#@title 6. Treinar (horas; salva no Drive e continua se cair)
import glob, os
ckpts = sorted(glob.glob(f"{TREINO}/**/checkpoints/*.ckpt", recursive=True), key=os.path.getmtime)
inicio = f"--ckpt_path '{ckpts[-1]}'" if ckpts else f"--model.warmstart_ckpt '{BASE}'"
print("Começando de:", ckpts[-1] if ckpts else "voz faber (primeira vez)")
!cd /content/piper1-gpl && python3 -m piper.train fit \
  --data.voice_name "{NOME_VOZ}" \
  --data.csv_path "{meta[0]}" \
  --data.audio_dir "{WAVS}" \
  --model.sample_rate 22050 \
  --data.espeak_voice "pt-br" \
  --data.cache_dir /content/cache \
  --data.config_path "{TREINO}/config.json" \
  --data.batch_size {LOTE} \
  --trainer.max_epochs {EPOCAS} \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.default_root_dir "{TREINO}" \
  {inicio}

In [ ]:
#@title 7. Gerar o arquivo da voz (.onnx)
import glob, os, shutil
ckpts = sorted(glob.glob(f"{TREINO}/**/checkpoints/*.ckpt", recursive=True), key=os.path.getmtime)
assert ckpts, "Nenhum checkpoint ainda: rode o passo 6."
SAIDA = f"{PASTA_DRIVE}/pt_BR-{NOME_VOZ}-medium.onnx"
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ckpts[-1]}" --output-file "{SAIDA}"
shutil.copy(f"{TREINO}/config.json", SAIDA + ".json")
print("Pronto! No seu Drive (pasta Lumi):", os.path.basename(SAIDA), "e", os.path.basename(SAIDA) + ".json")

In [ ]:
#@title 8. Ouvir um teste
!pip -q install piper-tts
!echo "Oi, chefe! Tudo certo por aí? Agora eu tenho uma voz só minha." | piper -m "{SAIDA}" -f /content/teste.wav
from IPython.display import Audio
Audio("/content/teste.wav")

## Colocar a voz na Sexta-feira

1. Baixe do Drive (pasta **Lumi**) os dois arquivos: `pt_BR-sexta-feira-medium.onnx` e `pt_BR-sexta-feira-medium.onnx.json`.
2. No app: **Ajustes → Voz offline (Piper) → ➕ Adicionar voz (.onnx)** e escolha o `.onnx` (o `.json` precisa estar na mesma pasta).
3. Em **Tipo de voz**, escolha **Voz offline (Piper)** — ou deixe o Google como principal: a voz Piper entra sozinha quando o Google não puder falar.

Achou a voz ainda "crua"? Aumente `EPOCAS` (ex.: 2000) e rode de novo os passos 1, 2, 3, 4, 5 e 6: ele continua do ponto salvo.